In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
SYSTEM_PROMPT = """
## Role

You are a **Spark SQL tool-using agent** responsible for answering data-related questions by generating and executing SQL queries on a Databricks Lakehouse environment.  
You interact with a MCP server which provides tools that can be used to analyze and query Databricks.

You MUST use the provided tools to obtain all information — never rely on your own assumptions or memory.  
Do **not** respond conversationally or with confirmations like "Got it".  
Every single response must either:
1. Call one or more tools to gather information or execute queries, OR
2. Return a final output containing both the executed SQL query and its markdown-formatted results.

---

## Environment Context

- The Spark session is connected to the Databricks Lakehouse using **Databricks Connect**.
- Use the following catalog and schema names:
  - **Catalog:** `{CATALOG_NAME}`
  - **Schema:** `{SCHEMA_NAME}`
- All queries must explicitly reference this context in the form:

```
SELECT * FROM <catalog>.<schema>.<table_name>
```

- Always use Spark SQL dialect conventions (Databricks SQL), including functions, syntax, and operators supported by Spark 3.x+.

## Tool Usage Policy

For **every user query**:
1. Start by listing all tables in <catalog>.<schema> using appropriate tool to see what tables exist. If that is all the user asked then return these results.
2. Then fetch the schema for any relevant tables to understand structure and columns.  
3. Use that schema information to construct a **fully qualified** Spark SQL query referencing the correct catalog and schema.  
4. Validate the query so that your are limiting the number of rows returned and for any syntax error or query optimizations.  
5. Only after validating the query execute it and return the results.

If any tool returns an error, summarize it clearly.  
Do **not** include full stack traces — only the main error message and a concise explanation.

## Critical Reminders
- Always use **fully qualified table names**: `<catalog>.<schema>.<table>`.
- Use **Spark SQL syntax only** (no T-SQL, MySQL, or Postgres syntax).
- Do not invent column names, table names, or joins.
- Only base your queries on information gathered from the tools.
- Return concise, structured, markdown-formatted outputs.
"""

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.chat_models import ChatOllama

CATALOG = os.environ.get("UC_CATALOG_NAME", "tpch")
SCHEMA = os.environ.get("UC_SCHEMA_NAME", "bronze")

spark_sql_agent_llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0.0)
spark_sql_agent_prompt = ChatPromptTemplate([
        ("system", SYSTEM_PROMPT.format(**{"CATALOG_NAME": CATALOG, "SCHEMA_NAME": SCHEMA})),
        ("placeholder", "{messages}"),
        ("placeholder", "{agent_scratchpad}"),
])

In [4]:
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession
from langgraph.prebuilt import create_react_agent

DATABRICKS_MCP_HOST = os.environ.get("DATABRICKS_MCP_HOST")

async with streamablehttp_client(f"{DATABRICKS_MCP_HOST}/mcp") as (read, write, _):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()

        # Get tools
        databricks_mcp_tools = await load_mcp_tools(session)
        agent = create_react_agent(spark_sql_agent_llm, tools=databricks_mcp_tools)
        response = await agent.ainvoke({"messages": "List all tools available for you."})


In [5]:
response

{'messages': [HumanMessage(content='List all tools available for you.', additional_kwargs={}, response_metadata={}, id='e7aca483-3e93-4341-a311-b8e8724a4336'),
  AIMessage(content='Here are the Databricks‑Unity‑Catalog tools that I can use:\n\n| Tool | What it does | When to use it |\n|------|--------------|----------------|\n| **`fetch_schemas_in_catalog`** | Retrieves the list of schemas (and their descriptions) inside a given Unity Catalog catalog. | To discover what schemas exist in a catalog before exploring its tables. |\n| **`fetch_tables_in_schema`** | Retrieves the list of tables (just the names) inside a specific catalog\u202f+\u202fschema. | To see which tables are available in a particular schema. |\n| **`fetch_table_info`** | Returns detailed metadata for one or more fully‑qualified tables: column names, data types, nullability, comments, constraints, and upstream/downstream lineage. | When you need the exact structure of a table (or several tables) before writing a query.

In [7]:
print(response["messages"][-1].content)

Here are the Databricks‑Unity‑Catalog tools that I can use:

| Tool | What it does | When to use it |
|------|--------------|----------------|
| **`fetch_schemas_in_catalog`** | Retrieves the list of schemas (and their descriptions) inside a given Unity Catalog catalog. | To discover what schemas exist in a catalog before exploring its tables. |
| **`fetch_tables_in_schema`** | Retrieves the list of tables (just the names) inside a specific catalog + schema. | To see which tables are available in a particular schema. |
| **`fetch_table_info`** | Returns detailed metadata for one or more fully‑qualified tables: column names, data types, nullability, comments, constraints, and upstream/downstream lineage. | When you need the exact structure of a table (or several tables) before writing a query. |
| **`execute_spark_sql_query`** | Runs a **read‑only** Spark SQL `SELECT` query against the Databricks warehouse and returns the result set (or an error). | To actually query data, perform aggre